<a href="https://colab.research.google.com/github/rxphaelbihag/Synthetic-Data-Generation/blob/main/Data_Loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading for TimeGAN Implementation

This notebook outlines the preprocessing of the dataset to be used for Synthetic Data Generation using TimeGAN introduced by Jinsung Yoon, Daniel Jarrett, and Mihaela van der Schaar in 2019.

The dataset to be used for this project is from Project CCHAIN available on Kaggle.

**"**
> The Project Climate Change, Health, and Artificial Intelligence (Project CCHAIN) dataset is a validated, open-sourced linked dataset containing 20 years (2003-2022) of climate, environmental, socioeconomic, and health dimensions at the barangay (village) level across twelve Philippine cities (Dagupan, Palayan, Navotas, Mandaluyong, Muntinlupa, Legazpi, Iloilo, Mandaue, Tacloban, Zamboanga, Cagayan de Oro, Davao).

**"**

Particularly, we will be looking at generating synthetic dataset of Weather Proxies and Air Pollutants.

**Focused Datasets:**
- climate_air_quality.csv (For Air Pollutants)
- climate_atmosphere.csv (For Weather Proxies)

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

import kagglehub
from kagglehub import KaggleDatasetAdapter

print("Successfully loaded required libraries!")

Successfully loaded required libraries!


## Load data from Kaggle

In [2]:
# @title
from google.colab import userdata
import os

# Set up Kaggle credentials
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

# Download the dataset
!kaggle datasets download -d thinkdatasci/project-cchain

# List contents to verify filenames if needed
# !unzip -l "project-cchain.zip"

# Define the two target CSV files
target_csv_1 = "climate_air_quality.csv"
target_csv_2 = "climate_atmosphere.csv"  # Change this to your second CSV filename

# Extract the first file if it doesn't exist
if not os.path.exists(target_csv_1):
    !unzip "project-cchain.zip" "$target_csv_1"
    print(f"Successfully unzipped '{target_csv_1}'.")
else:
    print(f"'{target_csv_1}' already exists. Skipping unzip.")

# Extract the second file if it doesn't exist
if not os.path.exists(target_csv_2):
    !unzip "project-cchain.zip" "$target_csv_2"
    print(f"Successfully unzipped '{target_csv_2}'.")
else:
    print(f"'{target_csv_2}' already exists. Skipping unzip.")

Dataset URL: https://www.kaggle.com/datasets/thinkdatasci/project-cchain
License(s): Attribution 4.0 International (CC BY 4.0)
project-cchain.zip: Skipping, found more recently modified local copy (use --force to force download)
'climate_air_quality.csv' already exists. Skipping unzip.
'climate_atmosphere.csv' already exists. Skipping unzip.


In [3]:
# @title
clim_airqual_df = pd.read_csv(target_csv_1)
clim_atmo_df = pd.read_csv(target_csv_2)

print("Successfully converted to pandas dataframes!")

Successfully converted to pandas dataframes!


## Data Inspection

In [4]:
# @title
datasets_info = {
    'Weather Data': clim_atmo_df.shape,
    'Pollutants Data': clim_airqual_df.shape,
}

print("Dataset dimensions (rows × columns):")
print("-" * 40)
for name, shape in datasets_info.items():
    print(f"{name:<30}: {shape[0]:,} × {shape[1]}")

Dataset dimensions (rows × columns):
----------------------------------------
Weather Data                  : 6,421,095 × 13
Pollutants Data               : 6,421,095 × 10


### First Few Records

In [5]:
clim_airqual_df.head()

,uuid,adm4_pcode,date,freq,no2,co,so2,o3,pm10,pm25
0,CAIRQ000000,PH015518001,2003-01-02,D,3.66,0.1034,0.31,38.39,23.63,16.21
1,CAIRQ000001,PH015518001,2003-01-03,D,4.35,0.1010,0.70,38.37,34.43,23.82
2,CAIRQ000002,PH015518001,2003-01-04,D,3.82,0.1016,0.34,37.72,27.76,19.16
3,CAIRQ000003,PH015518001,2003-01-05,D,3.07,0.1053,0.24,38.44,31.26,20.64
4,CAIRQ000004,PH015518001,2003-01-06,D,3.02,0.0953,0.22,39.89,24.70,16.68


In [6]:
clim_atmo_df.head()

,uuid,adm4_pcode,date,freq,tave,tmin,tmax,heat_index,pr,wind_speed,rh,solar_rad,uv_rad
0,CATMS000000,PH015518001,2003-01-01,D,26.85,24.49,30.17,27.91,0.0,2.00,68.59,193.62,23.77
1,CATMS000001,PH015518001,2003-01-02,D,25.95,23.13,29.70,26.34,0.0,4.54,60.56,211.05,24.21
2,CATMS000002,PH015518001,2003-01-03,D,25.07,22.49,28.59,25.67,0.0,1.53,68.47,199.69,23.15
3,CATMS000003,PH015518001,2003-01-04,D,25.55,22.11,29.81,25.90,0.0,3.10,61.32,222.02,24.83
4,CATMS000004,PH015518001,2003-01-05,D,25.35,22.68,29.33,25.72,0.0,4.23,60.50,203.69,23.48


## Data Filtering

The data available covers multiple Philippine cities at Barangay Level (ADM4). For this project, we will only be focusing on one barangay: Barangay Agdao, Davao City. This barangay is mapped to the `adm4_pcode=PH112402002`.

Moreover, we will only be using the following weather proxies: Average Temperature, Precipitation, Wind Speed, Relative Humidity, and Solar Radiation.

For the TimeGAN processing, the date column is also dropped so we will be filtering that out.

In [7]:
agdao_clim_atmo_df = clim_atmo_df.query('adm4_pcode == "PH112402002"')[['tave', 'pr', 'wind_speed', 'rh', 'solar_rad']]
agdao_clim_airqual_df = clim_airqual_df.query('adm4_pcode == "PH112402002"')[['no2', 'co', 'so2', 'o3', 'pm10', 'pm25']]

In [8]:
agdao_clim_atmo_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7305 entries, 4726335 to 4733639
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   tave        7305 non-null   float64
 1   pr          7305 non-null   float64
 2   wind_speed  7305 non-null   float64
 3   rh          7305 non-null   float64
 4   solar_rad   7305 non-null   float64
dtypes: float64(5)
memory usage: 342.4 KB


In [9]:
agdao_clim_airqual_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7305 entries, 4725688 to 6420863
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   no2     7304 non-null   float64
 1   co      7305 non-null   float64
 2   so2     7305 non-null   float64
 3   o3      7305 non-null   float64
 4   pm10    7304 non-null   float64
 5   pm25    7304 non-null   float64
dtypes: float64(6)
memory usage: 399.5 KB


## Null Values Handling

As noted by Project CCHAIN documentation, there is no values for 2003-01-01. This is depicted in the non-null count in the Air Quality dataframe. For this, we will be filling the missing gap using linear interpolation.

In [10]:
# Filter out entirely clean columns from printed output
null_cols = agdao_clim_airqual_df.columns[agdao_clim_airqual_df.isna().any()]
print(agdao_clim_airqual_df[agdao_clim_airqual_df.isna().any(axis=1)][null_cols])

         no2  pm10  pm25
6420863  NaN   NaN   NaN


In [11]:
# Fill any missing gaps in the timeline cleanly
agdao_clim_airqual_df = agdao_clim_airqual_df.interpolate(method='linear')
agdao_clim_airqual_df = agdao_clim_airqual_df.fillna(method='bfill').fillna(method='ffill')

/tmp/ipykernel_2653/1381491010.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  agdao_clim_airqual_df = agdao_clim_airqual_df.fillna(method='bfill').fillna(method='ffill')


In [12]:
agdao_clim_airqual_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7305 entries, 4725688 to 6420863
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   no2     7305 non-null   float64
 1   co      7305 non-null   float64
 2   so2     7305 non-null   float64
 3   o3      7305 non-null   float64
 4   pm10    7305 non-null   float64
 5   pm25    7305 non-null   float64
dtypes: float64(6)
memory usage: 399.5 KB


## Save to CSV
Now that the datasets are ready for analysis, we can save them to CSV for them to proceed to the TimeGAN implementation in a separate notebook.

In [13]:
agdao_clim_atmo_df.to_csv('brgy_agdao_weather.csv', index=False)
agdao_clim_airqual_df.to_csv('brgy_agdao_pollutants.csv', index=False)